# IDP based agents

> Agents that knows the underlying task and the optimal action

In [ ]:
#| default_exp agents.dynamic_pricing.inventory_constrained.IDP

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

from abc import ABC, abstractmethod
from typing import Union, Optional, List
import numpy as np
import joblib
import os
import statsmodels.api as sm
from ddopai.agents.dynamic_pricing.utils import GLMLink
from ddopai.envs.base import BaseEnvironment
from ddopai.agents.dynamic_pricing.mushroom_rl import PricingMushroomBaseAgent
from mushroom_rl.core import Agent
from ddopai.utils import MDPInfo
from ddopai.agents.obsprocessors import FlattenTimeDimNumpy
from ddopai.envs.actionprocessors import ClipAction

In [ ]:
#| export
class IDPPolicy():
    def __init__(self,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] = None,
                 actionprocessors: Optional[List[object]] = None,
                 agent_name: str | None = None,
                 task: dict = None,
                 price_function = None,
                 g = None,
                 ):
        
        alpha = np.array(task["alpha"])
        beta = np.array(task["beta"])
        assert type(alpha) == type(beta), "alpha and beta must be of the same type"
        if type(alpha) == None:
            alpha = np.zeros(environment_info.observation_space['features'].shape[0])   
            beta = np.zeros(environment_info.observation_space['features'].shape[0])
        self.environment_info = environment_info
        self.task = task
        self.T = task["horizon"]
        self.alpha = alpha
        self.beta = beta
        if environment_info.observation_space['features'].shape[0] == 1:
            self.E_X = np.array([1])
        else:
            self.E_X = np.full(environment_info.observation_space['features'].shape[0], 1 / (2 * np.sqrt(environment_info.observation_space['features'].shape[0])))
        self.actionprocessors = actionprocessors
        self.price_function = price_function # Needs to return an np array
        self.g = g
        self.t = 0
        self.mode = "train"
        self.actionprocessors.append(ClipAction(environment_info.action_space.low, environment_info.action_space.high))

    def draw_action(self, observation: np.ndarray):
        X = observation['features']
        B_t = observation['inventory']
        price = self.price_function(X, self.alpha, self.beta)
        lagrangian = self.lagrangian(B_t)
        price = price + lagrangian
        for processor in self.actionprocessors:
            price = processor(price)
        
        return np.array(price)
    
    def lagrangian(self, B_t):
        """
        Lagrangian function for the pricing problem
        """
        avg_remaining_B = (2 * B_t) / (self.T - self.t +1) 
        lagrangian = (avg_remaining_B - np.dot(self.alpha, self.E_X)) / np.dot(self.beta, self.E_X)
        return lagrangian
    
    def update_task(self, env):
        self.environment_info = env.mdp_info
        self.task = env.get_task()
        
        self.alpha = np.array(self.task["alpha"])
        self.beta = np.array(self.task["beta"])
        if self.environment_info.observation_space['features'].shape[0] == 1:
            self.E_X = np.array([1])
        else:
            self.E_X = np.full(self.environment_info.observation_space['features'].shape[0], 1 / (2 * np.sqrt(self.environment_info.observation_space['features'].shape[0])))
        self.T = self.task["horizon"]
        self.t = 0
        
        """TODO add change in price function"""
    def fit(self, X, Y, action):
        self.t += 1
    
        
    def reset(self):
        pass
        

In [ ]:
#| export
class IDPCoreAgent(Agent):

    """
    Base class for clairvoyant agents.
    """

    def __init__(self,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] = [],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 task: dict = None,
                 price_function = None,
                 g = None,
                 ):
        
        policy = IDPPolicy(environment_info=environment_info, obsprocessors=obsprocessors, actionprocessors=actionprocessors, task=task, price_function=price_function, g=g)
        self.agent_name = agent_name
        super().__init__(environment_info, policy)
        
    def fit(self, dataset, **kwargs):
        X = dataset[0][0]["features"]
        Y = kwargs["demand"][0]
        action = dataset[0][1]
        self.policy.fit(X, Y, action)
        
    def update_task(self, env):
        self.policy.update_task(env)
        
    
        

In [ ]:
#| export
class IDPAgent(PricingMushroomBaseAgent):
    """
    Wrapper class for IDPCoreAgent to interact with MushroomRL.
    """
    def __init__(self,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] =[],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 task: dict = None,
                 price_function = None,
                 g = None,
                 ):
        self.agent = IDPCoreAgent(environment_info = environment_info,
                                     obsprocessors = obsprocessors, 
                                     actionprocessors = actionprocessors, 
                                     agent_name = agent_name, 
                                     task=task,
                                     price_function = price_function, 
                                     g = g)
        super().__init__(environment_info = environment_info, obsprocessors = obsprocessors, agent_name = agent_name)
    def update_task(self, env: object):
        """ Update the environment specific parameters of the agent """
        self.agent.update_task(env)
        